# **Notebook 04 — Feature Engineering**

## Objectives

* Engineer new features from Time and Amount based on findings from Notebook 02
* Investigate and validate potential transformations
* Apply scaling appropriate for outlier-heavy data
* Apply SMOTE oversampling to training data only

## Inputs

* `outputs/v1/X_train.csv`, `outputs/v1/X_test.csv`
* `outputs/v1/y_train.csv`, `outputs/v1/y_test.csv`

## Outputs

* `outputs/v1/X_train_engineered.csv`, `outputs/v1/X_test_engineered.csv`
* `outputs/v1/X_train_resampled.csv`, `outputs/v1/y_train_resampled.csv`
* `outputs/v1/robust_scaler.pkl`
* Documentation of all transformation decisions

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\name\Desktop\All IN\Code Inst. Projects\credit-card-fraud-detection


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import matplotlib.pyplot as plt

X_train = pd.read_csv("outputs/v1/X_train.csv")
X_test = pd.read_csv("outputs/v1/X_test.csv")
y_train = pd.read_csv("outputs/v1/y_train.csv").squeeze()
y_test = pd.read_csv("outputs/v1/y_test.csv").squeeze()

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")
print(f"Features:  {X_train.shape[1]}")

Train set: (226980, 30)
Test set:  (56746, 30)
Features:  30


---
## 1. Engineer New Features

Based on findings from the Data Visualisation notebook:
- **H2 validated:** Fraud rate varies by hour → create `Hour` and `Is_Night` features
- **Amount distribution:** Highly skewed → create `Amount_log` 
- **H3 validated:** V14, V12, V10 are top discriminators → create interaction features
- **PCA variance:** Aggregate statistics across PCA components may capture anomalies

In [3]:
# Time-based features
# Hour derived from Time (seconds since first transaction)
X_train['Hour'] = (X_train['Time'] / 3600) % 24
X_test['Hour'] = (X_test['Time'] / 3600) % 24

# Night flag — fraud patterns may differ at night
X_train['Is_Night'] = ((X_train['Hour'] >= 22) | (X_train['Hour'] <= 5)).astype(int)
X_test['Is_Night'] = ((X_test['Hour'] >= 22) | (X_test['Hour'] <= 5)).astype(int)

print("Time-based features created: Hour, Is_Night")
print(f"  Night transactions in train: {X_train['Is_Night'].sum():,} "
      f"({X_train['Is_Night'].mean():.2%})")

Time-based features created: Hour, Is_Night
  Night transactions in train: 37,603 (16.57%)


In [4]:
# Amount-based features
# Log transform to reduce skewness
X_train['Amount_log'] = np.log1p(X_train['Amount'])
X_test['Amount_log'] = np.log1p(X_test['Amount'])

print("Amount feature created: Amount_log")
print(f"  Amount skewness before: {skew(X_train['Amount']):.2f}")
print(f"  Amount_log skewness:    {skew(X_train['Amount_log']):.2f}")

Amount feature created: Amount_log
  Amount skewness before: 14.39
  Amount_log skewness:    0.16


In [5]:
# Interaction features — top discriminating PCA components
# V14 and V12 are the two strongest fraud signals from H3
X_train['V14_x_V12'] = X_train['V14'] * X_train['V12']
X_test['V14_x_V12'] = X_test['V14'] * X_test['V12']

# V14 and V10 — second interaction between strong signals
X_train['V14_x_V10'] = X_train['V14'] * X_train['V10']
X_test['V14_x_V10'] = X_test['V14'] * X_test['V10']

print("Interaction features created: V14_x_V12, V14_x_V10")

Interaction features created: V14_x_V12, V14_x_V10


In [6]:
# Statistical aggregate features across all PCA components
# These capture overall "shape" of each transaction in PCA space
v_cols = [f'V{i}' for i in range(1, 29)]

X_train['V_mean'] = X_train[v_cols].mean(axis=1)
X_test['V_mean'] = X_test[v_cols].mean(axis=1)

X_train['V_std'] = X_train[v_cols].std(axis=1)
X_test['V_std'] = X_test[v_cols].std(axis=1)

X_train['V_skew'] = X_train[v_cols].skew(axis=1)
X_test['V_skew'] = X_test[v_cols].skew(axis=1)

print("Aggregate features created: V_mean, V_std, V_skew")
print(f"\nTotal features now: {X_train.shape[1]} (was 30)")

Aggregate features created: V_mean, V_std, V_skew

Total features now: 38 (was 30)


### Feature Engineering Rationale

| Feature | Type | Rationale |
|---------|------|-----------|
| Hour | Time-based | Captures time-of-day fraud patterns validated in H2 |
| Is_Night | Binary flag | Night hours may have different fraud dynamics |
| Amount_log | Transform | Reduces extreme skewness of Amount distribution |
| V14_x_V12 | Interaction | Non-linear relationship between the two strongest fraud signals |
| V14_x_V10 | Interaction | Second interaction between strong discriminators |
| V_mean | Aggregate | Average PCA value per transaction — anomalies may differ |
| V_std | Aggregate | Variance across PCA components — fraud may show unusual spread |
| V_skew | Aggregate | Distribution shape — fraud transactions may have different skew patterns |

---
## 2. Investigate Transformations

Checking whether additional transformations on Amount improve its distribution for modelling.

In [7]:
from sklearn.preprocessing import PowerTransformer

# Compare transformations
print("Amount Transformation Comparison")
print("=" * 50)
print(f"  Raw Amount skewness:        {skew(X_train['Amount']):.2f}")
print(f"  Log(1+Amount) skewness:     {skew(X_train['Amount_log']):.2f}")

# Yeo-Johnson can handle zeros and negatives
pt = PowerTransformer(method='yeo-johnson')
amount_pt = pt.fit_transform(X_train[['Amount']])
print(f"  Yeo-Johnson skewness:       {skew(amount_pt.flatten()):.2f}")

Amount Transformation Comparison
  Raw Amount skewness:        14.39
  Log(1+Amount) skewness:     0.16
  Yeo-Johnson skewness:       0.02


### Transformation Decision

- Log transform reduces Amount skewness substantially
- Yeo-Johnson further normalises but adds complexity
- **Decision:** Keep `Amount_log` as an engineered feature. Apply `RobustScaler` to all features that need scaling rather than PowerTransformer, because RobustScaler is more appropriate for outlier-heavy distributions and does not assume normality.

---
## 3. Feature Selection Analysis

Using Mutual Information to verify that all engineered features contribute positively to the classification task.

In [8]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train, y_train, random_state=42)

mi_df = pd.DataFrame({
    'feature': X_train.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print("Top 15 features by Mutual Information with fraud class:")
print("-" * 45)
for _, row in mi_df.head(15).iterrows():
    bar = "█" * int(row['mi_score'] * 100)
    print(f"  {row['feature']:15s}  {row['mi_score']:.4f}  {bar}")

# Check engineered features specifically
print("\nEngineered features MI scores:")
engineered = ['Hour', 'Is_Night', 'Amount_log', 'V14_x_V12',
              'V14_x_V10', 'V_mean', 'V_std', 'V_skew']
for feat in engineered:
    score = mi_df[mi_df['feature'] == feat]['mi_score'].values[0]
    print(f"  {feat:15s}  {score:.4f}")

Top 15 features by Mutual Information with fraud class:
---------------------------------------------
  V14_x_V10        0.0085  
  V14_x_V12        0.0082  
  V17              0.0080  
  V14              0.0079  
  V10              0.0073  
  V12              0.0073  
  V11              0.0066  
  V_mean           0.0062  
  V16              0.0059  
  V_std            0.0052  
  V3               0.0047  
  V4               0.0046  
  Is_Night         0.0042  
  V9               0.0041  
  V18              0.0040  

Engineered features MI scores:
  Hour             0.0011
  Is_Night         0.0042
  Amount_log       0.0012
  V14_x_V12        0.0082
  V14_x_V10        0.0085
  V_mean           0.0062
  V_std            0.0052
  V_skew           0.0011


### Feature Selection Decision

All engineered features show positive mutual information with the target variable, confirming they carry useful signal. No features are dropped — all contribute positively to the classification task.

---
## 4. Scaling

Applying RobustScaler to features that are not already on a standardised scale. V1-V28 are already scaled from PCA, so we only scale Amount, Time, and the engineered features.

In [9]:
from sklearn.preprocessing import RobustScaler
import joblib

# Only scale features not already standardised by PCA
cols_to_scale = ['Amount', 'Time', 'Hour', 'Amount_log',
                 'V14_x_V12', 'V14_x_V10', 'V_mean', 'V_std', 'V_skew']

scaler = RobustScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# Save scaler for the pipeline
joblib.dump(scaler, "outputs/v1/robust_scaler.pkl")

print("RobustScaler applied and saved.")
print(f"Scaled columns: {cols_to_scale}")

RobustScaler applied and saved.
Scaled columns: ['Amount', 'Time', 'Hour', 'Amount_log', 'V14_x_V12', 'V14_x_V10', 'V_mean', 'V_std', 'V_skew']


---
## 5. SMOTE Oversampling (Training Data Only)

Applying SMOTE to generate synthetic fraud samples in the training set. This is done **after** the train/test split to prevent data leakage — the test set must remain untouched to give an honest evaluation.

In [10]:
! pip install imbalanced-learn

In [11]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print(f"  Legitimate: {(y_train == 0).sum():,}")
print(f"  Fraud:      {(y_train == 1).sum():,}")
print(f"  Ratio:      {(y_train == 0).sum() // (y_train == 1).sum()}:1")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Legitimate: {(y_train_resampled == 0).sum():,}")
print(f"  Fraud:      {(y_train_resampled == 1).sum():,}")
print(f"  Ratio:      1:1")

Before SMOTE:
  Legitimate: 226,602
  Fraud:      378
  Ratio:      599:1

After SMOTE:
  Legitimate: 226,602
  Fraud:      226,602
  Ratio:      1:1


In [12]:
# Also test ADASYN for comparison and document the decision
from imblearn.over_sampling import ADASYN

adasyn = ADASYN(random_state=42)
X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)

print("SMOTE vs ADASYN Comparison")
print("=" * 50)
print(f"SMOTE result:  {pd.Series(y_train_resampled).value_counts().to_dict()}")
print(f"ADASYN result: {pd.Series(y_adasyn).value_counts().to_dict()}")

SMOTE vs ADASYN Comparison
SMOTE result:  {0: 226602, 1: 226602}
ADASYN result: {1: 226606, 0: 226602}


### SMOTE vs ADASYN Decision

- **SMOTE** produces exactly balanced classes by creating synthetic samples uniformly
- **ADASYN** generates slightly more samples near harder-to-learn instances

**Decision:** Using SMOTE for its predictability and consistent results. ADASYN produces a similar outcome and does not show meaningful improvement in preliminary tests. SMOTE's uniform approach is simpler to explain and reproduce.

---
## 6. Save Engineered Data

In [13]:
# Save engineered (scaled) train and test sets
X_train.to_csv("outputs/v1/X_train_engineered.csv", index=False)
X_test.to_csv("outputs/v1/X_test_engineered.csv", index=False)

# Save SMOTE-resampled training data
X_train_resampled.to_csv("outputs/v1/X_train_resampled.csv", index=False)
pd.Series(y_train_resampled, name='Class').to_csv(
    "outputs/v1/y_train_resampled.csv", index=False
)

print("Files saved to outputs/v1/:")
for f in sorted(os.listdir("outputs/v1")):
    size = os.path.getsize(f"outputs/v1/{f}") / (1024 * 1024)
    print(f"  {f}: {size:.1f} MB")

Files saved to outputs/v1/:
  X_test.csv: 28.6 MB
  X_test_engineered.csv: 37.6 MB
  X_train.csv: 114.4 MB
  X_train_engineered.csv: 150.3 MB
  X_train_resampled.csv: 306.0 MB
  robust_scaler.pkl: 0.0 MB
  y_test.csv: 0.2 MB
  y_train.csv: 0.6 MB
  y_train_resampled.csv: 1.3 MB


---
## Conclusions

* **8 new features** engineered: Hour, Is_Night, Amount_log, V14_x_V12, V14_x_V10, V_mean, V_std, V_skew
* **Total features** increased from 30 to 38
* All engineered features show positive **Mutual Information** with the target
* **RobustScaler** applied to non-PCA features — resistant to the outliers we kept
* **SMOTE** applied to training data only — balanced classes for modelling
* **ADASYN** tested as alternative — SMOTE selected for simplicity and consistency
* Test set remains **untouched** — no scaling leakage, no synthetic samples
